In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = "results"

seed_names = [
    "first_seed",
    "second_seed",
    "third_seed",
    "fourth_seed",
    "fifth_seed",
]

credibility_modes = [
    "discrete_credibility",
    "normally_distributed_credibility",
]

belief_types = {
    "type1": "Trimodal",
    "type2": "Asymmetric Shift Right",
    "type3": "Asymmetric Extremist",
    "type4": "Right Skewed"
}

regimes = ["mixed", "echo", "curated"]

METRIC_LABELS = {
    "mean_belief": "Mean Belief",
    "polarization_var": "Belief Variance (Polarization)",
    "share_extremes": "Share of Extremists",
    "assortativity": "Assortativity (Homophily)",
    "mean_influence": "Mean Influence",
    "var_influence": "Influence Variance",
    "mean_cred_weighted_influence": "Credibility-Weighted Influence",
    "influence_gini": "Influence Gini (Inequality)",
    "elite_fraction": "Top-10% Elite Fraction",
}

OUTPUT_DIR = "aggregated_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def plot_for_belief_and_cred(btype, cred_mode):

    fig, axes = plt.subplots(3, 3, figsize=(16, 12))
    axes = axes.flatten()

    for i, (metric, label) in enumerate(METRIC_LABELS.items()):
        ax = axes[i]

        for regime in regimes:
            dfs = []

            for seed in seed_names:
                file_path = os.path.join(RESULTS_DIR, seed, cred_mode, f"{btype}_{regime}_metrics.csv")
                if os.path.exists(file_path):
                    df = pd.read_csv(file_path)

                    # Keep only numeric columns so aggregation works
                    num_df = df.select_dtypes(include=[np.number])
                    dfs.append(num_df)

            if not dfs:
                continue

            combined = pd.concat(dfs, ignore_index=True)
            grouped = combined.groupby("step").agg(["mean", "std"])
            grouped.columns = ['_'.join(col) for col in grouped.columns]

            mean_col = f"{metric}_mean"
            std_col = f"{metric}_std"

            if mean_col in grouped.columns:
                ax.plot(grouped.index, grouped[mean_col], label=regime.capitalize(), linewidth=2)
                ax.fill_between(grouped.index,
                                grouped[mean_col] - grouped[std_col],
                                grouped[mean_col] + grouped[std_col],
                                alpha=0.15)

        ax.set_title(label)
        ax.set_xlabel("Step")
        ax.set_ylabel(label)
        ax.grid(alpha=0.3)
        ax.legend()

    fig.suptitle(f"{belief_types[btype]} — {cred_mode.replace('_', ' ').title()}", fontsize=18, y=1.02)
    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, f"{btype}_{cred_mode}.png")
    fig.savefig(out_path)
    plt.close()
    print(f"Saved: {out_path}")


for cred_mode in credibility_modes:
    for btype in belief_types:
        plot_for_belief_and_cred(btype, cred_mode)


Saved analysis\type1_discrete_credibility.png
Saved analysis\type2_discrete_credibility.png
Saved analysis\type3_discrete_credibility.png
Saved analysis\type4_discrete_credibility.png
Saved analysis\type1_normally_distributed_credibility.png
Saved analysis\type2_normally_distributed_credibility.png
Saved analysis\type3_normally_distributed_credibility.png
Saved analysis\type4_normally_distributed_credibility.png
